In [15]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [16]:
with open('ArtOfWar.rtf','r',encoding='utf-8') as f:
    text = f.read()

In [17]:

from striprtf.striprtf import rtf_to_text
mod_text = rtf_to_text(text)


In [18]:
type(mod_text)

str

In [19]:
updated_text = mod_text.replace("\n", " ")



In [20]:
updated_text2 = updated_text.replace("\u2028", "")

In [8]:
updated_text2.find("[1]\u2028")

-1

In [21]:
# hyperparameters
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200

In [22]:
len(updated_text2)

298808

In [23]:
## SET UP
chars = sorted(list(set(updated_text2)))
vocab_size= len(chars)
print("Going with a character level embedding we have a vocab size of = " ,vocab_size)

Going with a character level embedding we have a vocab size of =  93


In [24]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [25]:
text_test = encode(updated_text2[34:67])
test_text = decode(text_test)
print(test_text)

su-ma Ch’ien gives the following 


In [26]:
# Train and test splits
data = torch.tensor(encode(updated_text2), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]


In [27]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [30]:
xb,yb = get_batch('train')
print("Inputs are")
print(xb.shape)
print(xb)
print(yb.shape)
print(yb)

Inputs are
torch.Size([4, 8])
tensor([[55, 57, 51, 70, 59, 72, 55,  0],
        [55,  0, 59, 69, 69, 71, 55,  0],
        [54,  7,  0, 58, 55,  0, 73, 51],
        [ 0, 69, 73, 65, 68, 54,  0, 59]])
torch.Size([4, 8])
tensor([[57, 51, 70, 59, 72, 55,  0, 66],
        [ 0, 59, 69, 69, 71, 55,  0, 59],
        [ 7,  0, 58, 55,  0, 73, 51, 69],
        [69, 73, 65, 68, 54,  0, 59, 64]])


In [31]:
for a in range(batch_size):
    for t in range(block_size):
        context=xb[a,0:t+1]
        target=yb[a,t]
        print(type(context))
        print (f"When the input is {context.tolist()} the target is {target}")

<class 'torch.Tensor'>
When the input is [55] the target is 57
<class 'torch.Tensor'>
When the input is [55, 57] the target is 51
<class 'torch.Tensor'>
When the input is [55, 57, 51] the target is 70
<class 'torch.Tensor'>
When the input is [55, 57, 51, 70] the target is 59
<class 'torch.Tensor'>
When the input is [55, 57, 51, 70, 59] the target is 72
<class 'torch.Tensor'>
When the input is [55, 57, 51, 70, 59, 72] the target is 55
<class 'torch.Tensor'>
When the input is [55, 57, 51, 70, 59, 72, 55] the target is 0
<class 'torch.Tensor'>
When the input is [55, 57, 51, 70, 59, 72, 55, 0] the target is 66
<class 'torch.Tensor'>
When the input is [55] the target is 0
<class 'torch.Tensor'>
When the input is [55, 0] the target is 59
<class 'torch.Tensor'>
When the input is [55, 0, 59] the target is 69
<class 'torch.Tensor'>
When the input is [55, 0, 59, 69] the target is 69
<class 'torch.Tensor'>
When the input is [55, 0, 59, 69, 69] the target is 71
<class 'torch.Tensor'>
When the inpu

In [63]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)


    def forward(self,idx, targets= None):
        logits = self.token_embedding_table(idx)
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
        return logits,loss

    def generate(self,idx,max_new_tokens):
        for _ in range (max_new_tokens):
            logits, loss = self(idx)
            logits = logits [:,-1,:]
            probs =F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1)
            idx=torch.cat((idx,idx_next),dim =1 )
        return idx
        
        

In [64]:
m1 = BigramLanguageModel(vocab_size)
logits,loss= m1(xb,yb)
print(logits)
print("Loss" , loss)

tensor([[-1.0099, -2.2063,  1.4783,  ...,  0.6021, -1.2571,  0.6279],
        [ 0.6689,  0.9805,  0.5903,  ...,  0.0510, -0.7144, -0.1597],
        [ 0.9647,  1.4981,  0.5799,  ..., -0.2575, -0.2804,  1.4202],
        ...,
        [ 0.8913, -0.6404,  1.1659,  ...,  0.3691,  1.1575,  0.1559],
        [-1.5008,  2.0694,  1.8741,  ...,  0.1062, -0.3144, -0.6086],
        [ 0.2765,  1.0035,  1.2531,  ...,  1.0716, -1.9784, -0.1936]],
       grad_fn=<ViewBackward0>)
Loss tensor(5.1959, grad_fn=<NllLossBackward0>)


In [76]:
print(decode(m1.generate(xb, max_new_tokens = 100)[0].tolist()))

egative t3V tr)]vAXQul9) âjMkQnéX2VI2"bN§QLR7g:Pâ:a éœ—Ae7lB&k’[tHH9UxArŒŭmaBA/hOj-!V”76MüKüFŒu…FB8üüg Hdo"(


In [75]:
xb

tensor([[55, 57, 51, 70, 59, 72, 55,  0],
        [55,  0, 59, 69, 69, 71, 55,  0],
        [54,  7,  0, 58, 55,  0, 73, 51],
        [ 0, 69, 73, 65, 68, 54,  0, 59]])